# Data Preprocessing and Saving for Portfolio Optimization
## Objective
The goal of this notebook is to download historical stock price data, process it to calculate annualized expected returns and the covariance matrix, and save these key parameters for later use in optimization.
## Libraries Used
* `yfinance`: To download historical stock price data from Yahoo Finance.
* `numpy`: For numerical operations and array manipulation.
* `pandas`: For data manipulation, especially with time series and DataFrames.
* `pickle`: To save our processed Python objects (
    NumPy arrays, lists) to files, so we don't have to re-download data every time.
* `os`: To help manage file paths and create directories.
* `time`: To introduce small delays between batches of downloads.


## Imports and Setup
This cell imports all necessary libraries and defines the core data processing function.

In [1]:
pip install yfinance

Note: you may need to restart the kernel to use updated packages.


In [2]:
import yfinance as yf
import numpy as np
import pandas as pd
import time
import os
import pickle # Import the pickle module for saving/loading Python objects

In [3]:
def download_and_process_stock_data(all_tickers, start_date, end_date, batch_size=30):
    """
    Downloads historical prices, calculates daily returns,
    and computes mean returns and covariance matrix.
    Includes debugging prints.

    Args:
        all_tickers (list): A list of all stock ticker symbols.
        start_date (str): Start date for data download in 'YYYY-MM-DD' format.
        end_date (str): End date for data download in 'YYYY-MM-DD' format.
        batch_size (int): Number of tickers to download per batch.

    Returns:
        tuple: A tuple containing:
            - means (np.ndarray or None): Annualized mean daily returns for each stock, or None if failed.
            - cov_matrix (np.ndarray or None): Annualized covariance matrix of daily returns, or None if failed.
            - successful_tickers (list): List of tickers for which data was successfully downloaded and processed.
            - failed_tickers_details (dict): Dictionary mapping failed tickers to their error.
    """
    print(f"--- Initiating Data Download ---")
    print(f"Total tickers to process: {len(all_tickers)}")
    print(f"Date Range: {start_date} to {end_date}")
    print(f"Batch Size: {batch_size}")

    all_prices_data = pd.DataFrame()
    failed_tickers_details = {}
    successful_tickers = []
    
    
    # Stock data is slow and heavy to download, especially if you request hundreds of tickers at once!
    # That's why we will be using batches for: 
    # (1) more robust downloads, (2) avoid API rate limits, and (3) memory efficiency
    num_batches = (len(all_tickers) + batch_size - 1) // batch_size

    for i in range(num_batches):
        batch_tickers = all_tickers[i * batch_size : (i + 1) * batch_size]
        print(f"\nProcessing Batch {i+1}/{num_batches} ({len(batch_tickers)} tickers)...")

        try:
            # Fetch data for the current batch - IMPORTANT: Explicitly set auto_adjust=True to get adj_close
            # Adjusted prices = prices corrected for: (1) stock splits, (2) dividends, (3) mergers, or other corporate actions
            data = yf.download(
                batch_tickers,
                start=start_date,
                end=end_date,
                progress=False,
                auto_adjust=True # <<< IMPORTANT! SET THIS TO TRUE
            )

            if data.empty:
                print(f"Batch {i+1}: No data returned by yfinance for this batch.")
                for ticker in batch_tickers:
                    failed_tickers_details[ticker] = "No data returned by yfinance for the batch."
                continue

            # --- DEBUGGING START ---
            print(f"Batch {i+1}: yfinance returned data. Shape: {data.shape}")
            is_multi_index = isinstance(data.columns, pd.MultiIndex)
            print(f"Batch {i+1}: Columns structure is MultiIndex: {is_multi_index}")
            if is_multi_index:
                 print(f"Batch {i+1}: Columns Head: {list(data.columns)[:10]}...")
            else:
                 print(f"Batch {i+1}: Columns Head: {data.columns.tolist()[:10]}...")
            # --- DEBUGGING END ---

            current_batch_processed_tickers = []

            for ticker in batch_tickers:
                price_series = None

                if is_multi_index:
                    close_col_tuple = ('Close', ticker)
                    if close_col_tuple in data.columns:
                        price_series = data[close_col_tuple]
                    else:
                        failed_tickers_details[ticker] = "Close data not found for ticker in MultiIndex."
                        print(f"    FAILED {ticker}: Close data not found for ticker in MultiIndex.")
                else:
                    # Fallback for non-MultiIndex, though less expected with multiple tickers and auto_adjust=True
                    if 'Close' in data.columns:
                        # This path might be hit if yfinance returns a flat structure for some reason.
                        # For a single ticker download it might be 'Close', but for batches it's usually MultiIndex.
                        # If not MultiIndex, and 'Close' is a column, it might be the only data.
                        # We'll assume for now that if it's not MultiIndex, the 'Close' column is relevant.
                        # This part could be more robust for edge cases.
                        price_series = data['Close'] # Directly access 'Close' if it's a flat DataFrame
                    else:
                         failed_tickers_details[ticker] = "'Close' column not found in flat data."
                         print(f"    FAILED {ticker}: 'Close' column not found in flat data.")


                if price_series is not None:
                    if price_series.notna().all():
                        all_prices_data[ticker] = price_series
                        current_batch_processed_tickers.append(ticker)
                    else:
                        failed_tickers_details[ticker] = "Contains missing 'Close' values."
                        print(f"    FAILED {ticker}: Contains missing 'Close' values.")

            successful_tickers.extend(current_batch_processed_tickers)
            print(f"Batch {i+1}: Successfully processed {len(current_batch_processed_tickers)}/{len(batch_tickers)} tickers.")

        except Exception as e:
            print(f"Batch {i+1}: An uncaught error occurred during download or processing: {e}")
            for ticker in batch_tickers:
                if ticker not in failed_tickers_details:
                    failed_tickers_details[ticker] = str(e)

        if i < num_batches - 1:
            time.sleep(2)

    print(f"\n--- Data Download and Initial Processing Complete ---")
    print(f"Total tickers processed (successfully retrieved and with no missing data): {len(successful_tickers)}")
    print(f"Total tickers failed or had missing data: {len(failed_tickers_details)}")
    if failed_tickers_details:
        print(f"Sample of failed tickers: {list(failed_tickers_details.keys())[:5]}{'...' if len(failed_tickers_details) > 5 else ''}")

    if all_prices_data.empty or all_prices_data.shape[1] == 0:
        print("\nERROR: No valid price data found for any tickers after processing. Cannot proceed.")
        return None, None, [], {}

    print("\n--- Calculating Returns and Covariance ---")
    daily_returns = all_prices_data.pct_change().dropna()

    if daily_returns.shape[0] < 2:
        print(f"ERROR: Not enough data points ({daily_returns.shape[0]} rows) after calculating daily returns to compute covariance.")
        return None, None, successful_tickers, failed_tickers_details

    trading_days_per_year = 252
    mean_daily_returns = daily_returns.mean()
    annualized_mean_returns = mean_daily_returns * trading_days_per_year
    daily_cov_matrix = daily_returns.cov()
    annualized_cov_matrix = daily_cov_matrix * trading_days_per_year

    final_successful_tickers = list(daily_returns.columns)

    print(f"Successfully calculated statistics for {len(final_successful_tickers)} tickers.")
    print(f"Annualized Mean Returns shape: {annualized_mean_returns.shape}")
    print(f"Annualized Covariance Matrix shape: {annualized_cov_matrix.shape}")

    return annualized_mean_returns.values, annualized_cov_matrix.values, final_successful_tickers, failed_tickers_details

## Define Tickers and Date Range
Here, we define the universe of stocks and the historical period for our analysis. This ensures we have a consistent dataset for our optimization.



In [5]:
# --- Define Your Universe of 100 Stocks ---
# This is a curated list of generally reliable stocks to ensure data availability.

# In a real-world scenario, you'd use a broader, carefully selected list.

my_100_tickers =my_100_tickers = [
    'JD','CSCO','BAC','AEP','NVDA','JNJ','BABA','MCD','WFC','CSX',
    'HON','VIPS','MSFT','SYK','ECL','C','META','MDT','ORCL','COO',
    'SBUX','MS','MMC','CRWD','GS','CHWY','MMM','AAPL','SRE','JPM',
    'DLTR','LLY','DDOG','PYPL','GOOG','XOM','SO','CAT','ISRG','VRTX',
    'OKTA','FSLY','SNPS','IBM','ADBE','NKE','AMD','MDLZ','MELI','INTC',
    'COST','MRNA','NEE','SMCI','JMIA','ABT','GOOGL','EL','KO','CVX',
    'HD','PFE','MA','ASML','CVS','KDP','DXC','EQIX','ABBV','NFLX',
    'AMZN','DOV','MU','CRM','ETSY','QCOM','CB','ROKU','TM','PEP',
    'AMAT','PG','CMCSA','UNH','LOW','EMR','ZM','RTX','TXN','MCHP',
    'DHI','MRK','TSLA','DUK','CI','CPRT','BRK-B','DG','EFX','CTAS'
]

# Ensure exactly 100 tickers (adjust if your list is shorter or longer)
if len(my_100_tickers) > 100:
    my_100_tickers = my_100_tickers[:100]
elif len(my_100_tickers) < 100:
    print(f"Warning: Created a universe of {len(my_100_tickers)} tickers, which is less than the target of 100.")

print(f"Defined universe of {len(my_100_tickers)} tickers.")

# --- Define Date Range ---
# We use a fixed historical date range to ensure consistency and avoid future dates.
# This range is chosen because it's entirely in the past

start_date_fixed = '2024-04-15'
end_date_fixed = '2026-04-15'

print(f"Using FIXED historical date range: {start_date_fixed} to {end_date_fixed}")

Defined universe of 100 tickers.
Using FIXED historical date range: 2024-04-15 to 2026-04-15


## Execute Data Fetching and Processing
* This is the core step where we download the data and perform the calculations.
* This might take a few minutes depending on your internet connection and the number of tickers.

In [6]:
# --- Run the Data Fetching and Processing ---
print("Starting data fetching and processing. This may take a few minutes...")

expected_returns_np, covariance_matrix_np, successful_tickers, failed_tickers_details = download_and_process_stock_data(
    my_100_tickers, start_date_fixed, end_date_fixed, batch_size=25 # Adjust batch_size if you encounter errors
)

# Check if data processing was successful before proceeding to saving
if expected_returns_np is None or covariance_matrix_np is None or len(successful_tickers) == 0:
    print("\nData processing failed. Please review the output from the previous cells for specific errors.")
else:
    print("\nData preprocessing complete. Results are ready to be saved.")

Starting data fetching and processing. This may take a few minutes...
--- Initiating Data Download ---
Total tickers to process: 100
Date Range: 2024-04-15 to 2026-04-15
Batch Size: 25

Processing Batch 1/4 (25 tickers)...



1 Failed download:
['MMC']: YFTzMissingError('possibly delisted; no timezone found')


Batch 1: yfinance returned data. Shape: (501, 126)
Batch 1: Columns structure is MultiIndex: True
Batch 1: Columns Head: [('Adj Close', 'MMC'), ('Close', 'AEP'), ('Close', 'BABA'), ('Close', 'BAC'), ('Close', 'C'), ('Close', 'COO'), ('Close', 'CRWD'), ('Close', 'CSCO'), ('Close', 'CSX'), ('Close', 'ECL')]...
    FAILED MMC: Contains missing 'Close' values.
Batch 1: Successfully processed 24/25 tickers.

Processing Batch 2/4 (25 tickers)...
Batch 2: yfinance returned data. Shape: (501, 125)
Batch 2: Columns structure is MultiIndex: True
Batch 2: Columns Head: [('Close', 'AAPL'), ('Close', 'ADBE'), ('Close', 'AMD'), ('Close', 'CAT'), ('Close', 'CHWY'), ('Close', 'DDOG'), ('Close', 'DLTR'), ('Close', 'FSLY'), ('Close', 'GOOG'), ('Close', 'IBM')]...
Batch 2: Successfully processed 25/25 tickers.

Processing Batch 3/4 (25 tickers)...
Batch 3: yfinance returned data. Shape: (501, 125)
Batch 3: Columns structure is MultiIndex: True
Batch 3: Columns Head: [('Close', 'ABBV'), ('Close', 'ABT'), 

## Save the Processed Data
To avoid re-downloading and re-calculating every time, we save the results to files. This allows Notebook 2 to load them directly.

In [7]:
# --- Saving the Data ---
# Define the filenames for our saved data
returns_file = 'processed_data/expected_returns.pkl'
cov_matrix_file = 'processed_data/covariance_matrix.pkl'
tickers_file = 'processed_data/successful_tickers.pkl'
failed_tickers_file = 'processed_data/failed_tickers_details.pkl' # Also save failure info for reference

# Create a directory to store the files if it doesn't exist
# This helps keep your project organized.
os.makedirs('processed_data', exist_ok=True)

# Save each piece of data using pickle
try:
    with open(returns_file, 'wb') as f:
        pickle.dump(expected_returns_np, f)
    print(f"Saved expected returns to: {returns_file}")

    with open(cov_matrix_file, 'wb') as f:
        pickle.dump(covariance_matrix_np, f)
    print(f"Saved covariance matrix to: {cov_matrix_file}")

    with open(tickers_file, 'wb') as f:
        pickle.dump(successful_tickers, f)
    print(f"Saved successful tickers list to: {tickers_file}")

    if failed_tickers_details: # Only save if there were failures
        with open(failed_tickers_file, 'wb') as f:
            pickle.dump(failed_tickers_details, f)
        print(f"Saved failed tickers details to: {failed_tickers_file}")

    print("\nAll necessary data successfully saved! You can now proceed to Notebook 2 (Loading and Using Gurobi for Portfolio Opt).")

except Exception as e:
    print(f"\nAn error occurred while saving the data: {e}")
    print("Please ensure you have write permissions in the directory.")

Saved expected returns to: processed_data/expected_returns.pkl
Saved covariance matrix to: processed_data/covariance_matrix.pkl
Saved successful tickers list to: processed_data/successful_tickers.pkl
Saved failed tickers details to: processed_data/failed_tickers_details.pkl

All necessary data successfully saved! You can now proceed to Notebook 2 (Loading and Using Gurobi for Portfolio Opt).
